# Epileptic Seizure Detection — SCTN resonators vs FFT (analysis)

Mirrors the **Situational Awareness Classification** analysis notebook
(Hadad et al., 2024) — Figures 4, 6, 7 — adapted to **seizure vs non-seizure**
on the CHB-MIT dataset instead of focus / unfocus / drowsed.

The SCTN resonator spikes are **pre-encoded** on disk (produced by
`src/1.signal2spikes_chbmit.py`); this notebook only *visualises* them.
Set `DATA_ROOT` below to the project folder that contains `dataset/`.

Sections (exactly the paper's analysis sections):
1. Signal — time domain + spectrogram
2. FFT spectrogram (band bins)
3. Spikes domain (single resonator)
4. **FFT spectrogram vs SCTN spikegram** (Fig 4)
5. Per-channel FFT grid
6. Per-channel SCTN spikegram grid
7. Combined per-channel FFT + SCTN grid
8. **Topography — FFT vs Spikes per band** (Fig 6)
9. **Topography — seizure vs non-seizure** (Fig 7)


## Setup

In [ ]:
# --- Colab: install deps and mount the dataset from Google Drive ---
# !pip install sctn mne
# from google.colab import drive; drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import scipy
from scipy.signal import butter, filtfilt, spectrogram
import mne

In [ ]:
# ---------------- CONFIG ----------------
# Point DATA_ROOT at the project folder containing dataset/chbmit + dataset/spikes_raw.
# Local example:  DATA_ROOT = ".."          (running from the notebook/ folder)
# Colab example:  DATA_ROOT = "/content/drive/MyDrive/Epileptic-Seizure-Classification"
DATA_ROOT = ".."

CHBMIT_ROOT = f"{DATA_ROOT}/dataset/chbmit"
SPIKES_ROOT = f"{DATA_ROOT}/dataset/spikes_raw"

FS = 256                                   # CHB-MIT sampling rate

SUBJECT           = "chb01"
EDF_FILE          = "chb01_03.edf"
EDF_STEM          = "chb01_03"
SEIZURE_MINUTE    = 49                      # minute containing seizure onset (2996 s)
NON_SEIZURE_MINUTE = 4                      # interictal minute

CHANNELS = ["FT9-FT10", "T7-P7", "T8-P8", "P7-O1", "F7-T7",
            "P8-O2", "CZ-PZ", "P3-O1", "FP1-F7", "C4-P4"]

bands = {"Delta": (0.5, 4), "Theta": (4, 8), "Alpha": (8, 14),
         "Beta": (14, 32), "Gamma": (32, 62)}

# resonator banks — must match the encoder (1.signal2spikes_chbmit.py)
clk_resonators = {
    15360:  [1.1, 1.3, 1.6, 1.9, 2.2, 2.5, 2.88, 3.05, 3.39, 3.7,
             4.12, 4.62, 5.09, 5.45, 5.87, 6.36, 6.8, 7.6, 8.6],
    153600: [10.5, 11.5, 12.8, 16.6, 19.4, 22.0, 24.8, 28.4, 30.5,
             34.7, 37.2, 40.2, 43.2, 47.7, 52.6, 57.2],
}
MAIN_CLK = max(clk_resonators)
FULL_RESONATOR_FREQS = sum(clk_resonators.values(), [])
BAND_LABELS = [f"{lo}-{hi} ({b})" for b, (lo, hi) in bands.items()]

In [ ]:
# ---------------- HELPERS (adapted from the SA-paper notebook) ----------------
def bp_filter(data, f_lo, f_hi, fs):
    ny = fs / 2.0
    b1, a1 = butter(6, f_hi / ny, btype="lowpass")
    b2, a2 = butter(6, f_lo / ny, btype="highpass")
    return filtfilt(b2, a2, filtfilt(b1, a1, data, axis=0), axis=0)

def get_trial_data(edf_file=EDF_FILE):
    """Load one CHB-MIT recording, keep the 10 bipolar channels, bandpass 0.5-62 Hz."""
    raw = mne.io.read_raw_edf(f"{CHBMIT_ROOT}/{SUBJECT}/{edf_file}", preload=True, verbose="ERROR")
    avail = [c.strip() for c in raw.ch_names]
    picks, names = [], []
    for want in CHANNELS:
        w = want.upper().replace(" ", "")
        for a in avail:
            an = a.upper().replace(" ", "")
            if (an == w or an.startswith(w)) and a not in picks:
                picks.append(a); names.append(want); break
    data = bp_filter(raw.get_data(picks=picks).T, 0.5, 62.0, FS)
    return pd.DataFrame({n: data[:, i] for i, n in enumerate(names)})

def spikes_to_events(spikes):
    return np.where(spikes == 1)[0]

def load_channel_events(label, minute, ch):
    """{clk_freq: [event-index array per resonator f0]} for one (channel, minute)."""
    out = {clk: [] for clk in clk_resonators}
    for clk, freqs in clk_resonators.items():
        for f0 in freqs:
            p = f"{SPIKES_ROOT}/{SUBJECT}/{EDF_STEM}/{label}/{minute}/{ch}/{clk}/{f0}.npz"
            out[clk].append(spikes_to_events(np.load(p)["spikes"]))
    return out

def plot_spectrogram(data, channel_name, fs, fmin, fmax, show=True):
    f, t, Sxx = spectrogram(data, fs, noverlap=0, nperseg=fs)
    Sxx = Sxx[:, :-1]
    if show:
        plt.figure(figsize=(14, 5))
        plt.ylim(fmin, fmax); plt.xlabel("Time (s)"); plt.ylabel("Frequency (Hz)")
        plt.title(f"ch={channel_name} fs={fs} band=({fmin}-{fmax})Hz")
        plt.imshow(Sxx, aspect="auto", cmap="jet", origin="lower",
                   extent=[t[0], t[-1], f[0], f[-1]]); plt.colorbar()
    return Sxx

def fft_spectrogram_to_bins(Sxx, window, normalize_band=True):
    n = len(Sxx[0]) // window + 1
    out = np.zeros((len(bands), n))
    for bi, (lo, hi) in enumerate(bands.values()):
        alpha = (hi - lo) if normalize_band else 1
        for i in range(n):
            out[bi][i] = np.sum(Sxx[int(lo):int(hi), i * window:(i + 1) * window]) / alpha
    return out

def spikes_event_spectrogram(clk_freq, events, window_ms, duration_s):
    window = clk_freq / 1000 * window_ms
    N = int(np.ceil(duration_s * clk_freq / window))
    bins = np.zeros(N, dtype=int)
    if len(events):
        idx, cnt = np.unique((events // window).astype(int), return_counts=True)
        idx = idx[idx < N]; bins[idx] = cnt[:len(idx)]
    return bins

def events_to_max_spectrogram(resonators_by_clk, duration_s):
    arrays = sum(resonators_by_clk.values(), [])
    out = np.zeros((len(arrays), (duration_s * 1000) // 100))
    i = 0
    for clk, ev_list in resonators_by_clk.items():
        for events in ev_list:
            spec = spikes_event_spectrogram(clk, events, 10, duration_s)
            step = max(1, len(spec) // out.shape[1])
            out[i] = spec[: out.shape[1] * step].reshape(out.shape[1], step).max(axis=1)
            out[i] *= MAIN_CLK / clk
            out[i] -= out[i].mean(); out[i][out[i] < 0] = 0
            i += 1
    return out

def spikes_to_bands(spikes_spectrogram, freqs, normalize_band=True):
    freqs = np.array(freqs)
    out = np.zeros((len(bands), spikes_spectrogram.shape[1]))
    for bi, (lo, hi) in enumerate(bands.values()):
        m = (freqs >= lo) & (freqs < hi)
        out[bi] = spikes_spectrogram[m].sum(axis=0)
        if normalize_band and freqs[m].sum() > 0:
            out[bi] /= freqs[m].sum()
    return out

def plot_bins(Sxx, duration, labels, show=True, colorbar=True, gridlines=True,
              axs=None, fig=None, vmin=None, vmax=None, fontsize=None):
    if axs is None:
        fig, axs = plt.subplots(figsize=(14, 5))
    im = axs.imshow(Sxx, aspect="auto", cmap="jet", origin="lower",
                    extent=[0, duration, 0, len(Sxx)], vmin=vmin, vmax=vmax)
    axs.set_yticks(np.arange(.5, len(Sxx) + .5), labels=labels, fontsize=fontsize)
    if gridlines:
        axs.set_yticks(np.arange(0, len(labels)), minor=True)
        axs.grid(which="minor", color="grey", linestyle="-", linewidth=1)
    if colorbar:
        cbar = fig.colorbar(im, ax=axs); cbar.set_ticks([])
    if show:
        plt.show()
    return axs

## 1. Signal — time domain + spectrogram

In [ ]:
example_data = get_trial_data(EDF_FILE)
channel = "T7-P7"
t0, t1 = SEIZURE_MINUTE * 60, (SEIZURE_MINUTE + 1) * 60      # the seizure-onset minute
sliced = example_data[channel][t0 * FS:t1 * FS].to_numpy()

plt.figure(figsize=(14, 3))
plt.plot(np.linspace(0, t1 - t0, len(sliced)), sliced)
plt.title(f"Signal on time domain — {channel} (seizure minute)")
plt.xlabel("Seconds"); plt.show()

plt.figure(figsize=(14, 3)); plt.title("Spectrogram of the signal")
plt.specgram(sliced, Fs=FS); plt.show()

## 2. FFT spectrogram (band bins)

In [ ]:
fft_spec = plot_spectrogram(sliced, channel, FS, fmin=0.5, fmax=64, show=True)
fft_bins = fft_spectrogram_to_bins(fft_spec, window=1)
ax = plot_bins(fft_bins, len(sliced) / FS, BAND_LABELS, show=False)
plt.title("(a) FFT Spectrogram"); plt.xlabel("Time (s)"); plt.ylabel("Frequency band")
plt.show()

## 3. Spikes domain (single resonator)

In [ ]:
duration = 60                                     # one minute
ch_events = load_channel_events("seizure", SEIZURE_MINUTE, channel)
clk = 153600
events = ch_events[clk][0]                         # first fast-clock resonator

y = np.zeros(clk * duration); y[events[events < len(y)]] = 1
y = np.convolve(y, np.ones(500, dtype=int), "valid")
plt.figure(figsize=(14, 3))
plt.plot(np.linspace(0, duration, len(y)), y)
plt.title("Signal on spikes domain"); plt.xlabel("Seconds"); plt.ylabel("Spike density")
plt.show()

spec = spikes_event_spectrogram(clk, events, 10, duration)
one_s = spec[: (len(spec) // duration) * duration].reshape(duration, -1).max(axis=1)[None, :]
plot_bins(one_s, duration, labels=[f"{clk_resonators[clk][0]}"], show=False)
plt.title("Spikes in 1-s bins"); plt.xlabel("Seconds"); plt.show()

## 4. FFT spectrogram vs SCTN spikegram (paper Fig 4)

In [ ]:
max_spec = events_to_max_spectrogram(ch_events, duration)
spikes_bands = spikes_to_bands(max_spec, FULL_RESONATOR_FREQS)
spikes_bands = spikes_bands[:, :(spikes_bands.shape[1] // 10) * 10]
spikes_bands = spikes_bands.reshape(spikes_bands.shape[0], -1, 10).sum(axis=2)

fig, axs = plt.subplots(2, 1, figsize=(14, 8))
plot_bins(fft_bins, len(sliced) / FS, BAND_LABELS, show=False, fig=fig, axs=axs[0])
axs[0].set_title("(a) FFT spectrogram", fontsize=18); axs[0].set_ylabel("Frequency band")
plot_bins(spikes_bands, duration, BAND_LABELS, show=False, fig=fig, axs=axs[1])
axs[1].set_title("(b) SCTN spikegram", fontsize=18)
axs[1].set_xlabel("Time (s)"); axs[1].set_ylabel("Frequency band")
fig.tight_layout(); plt.show()

## 5. Per-channel FFT spectrogram grid

In [ ]:
fft_bins_by_ch = {}
for ch in CHANNELS:
    s = example_data[ch][t0 * FS:t1 * FS].to_numpy()
    fft_bins_by_ch[ch] = fft_spectrogram_to_bins(
        plot_spectrogram(s, ch, FS, 0.5, 64, show=False), window=30, normalize_band=False)

fig, axs = plt.subplots(5, 2, figsize=(14, 14))
for i, ch in enumerate(CHANNELS):
    plot_bins(fft_bins_by_ch[ch], duration, BAND_LABELS, show=False, colorbar=False,
              fig=fig, axs=axs[i // 2][i % 2])
    axs[i // 2][i % 2].set_title(ch)
plt.suptitle("FFT spectrogram (per channel, seizure minute)"); plt.tight_layout(); plt.show()

## 6. Per-channel SCTN spikegram grid

In [ ]:
def channel_spikegram(label, minute, ch):
    ev = load_channel_events(label, minute, ch)
    m = events_to_max_spectrogram(ev, 60)
    b = spikes_to_bands(m, FULL_RESONATOR_FREQS, normalize_band=False)
    b = b[:, :(b.shape[1] // 10) * 10].reshape(b.shape[0], -1, 10).sum(axis=2)
    return b

spikegram_by_ch = {ch: channel_spikegram("seizure", SEIZURE_MINUTE, ch) for ch in CHANNELS}
vmax = max(v.max() for v in spikegram_by_ch.values())

fig, axs = plt.subplots(5, 2, figsize=(14, 14))
for i, ch in enumerate(CHANNELS):
    plot_bins(spikegram_by_ch[ch], duration, BAND_LABELS, show=False, colorbar=False,
              fig=fig, axs=axs[i // 2][i % 2], vmin=0, vmax=vmax)
    axs[i // 2][i % 2].set_title(ch)
plt.suptitle("SCTN spikegram (per channel, seizure minute)"); plt.tight_layout(); plt.show()

## 7. Combined per-channel FFT + SCTN grid

In [ ]:
fig, axs = plt.subplots(5, 4, figsize=(22, 14))
for i, ch in enumerate(CHANNELS):
    r = i // 2
    cs, cf = (0, 1) if i % 2 == 0 else (2, 3)
    labels = BAND_LABELS if i % 2 == 0 else []
    plot_bins(spikegram_by_ch[ch], duration, labels, show=False, colorbar=False,
              fig=fig, axs=axs[r][cs], fontsize=11)
    axs[r][cs].set_title(f"SCTN spikegram — {ch}", fontsize=11)
    plot_bins(fft_bins_by_ch[ch], duration, [], show=False, colorbar=False,
              fig=fig, axs=axs[r][cf], fontsize=11)
    axs[r][cf].set_title(f"FFT spectrogram — {ch}", fontsize=11)
plt.suptitle("SCTN vs FFT per channel (seizure minute)"); plt.tight_layout(); plt.show()

## 8. Topography — FFT vs Spikes per band (paper Fig 6)

In [ ]:
# bipolar-pair midpoint coordinates on a normalized head
ELECTRODE_XY = {
    "FP1": (-0.27, 0.95), "FP2": (0.27, 0.95), "F7": (-0.81, 0.59), "F3": (-0.55, 0.55),
    "FZ": (0, 0.55), "F4": (0.55, 0.55), "F8": (0.81, 0.59), "T7": (-0.95, 0), "C3": (-0.5, 0),
    "CZ": (0, 0), "C4": (0.5, 0), "T8": (0.95, 0), "P7": (-0.81, -0.59), "P3": (-0.55, -0.55),
    "PZ": (0, -0.55), "P4": (0.55, -0.55), "P8": (0.81, -0.59), "O1": (-0.27, -0.95),
    "O2": (0.27, -0.95), "FT9": (-0.88, 0.2), "FT10": (0.88, 0.2),
}
def channel_xy(ch):
    a, b = ch.split("-"); ax, ay = ELECTRODE_XY[a]; bx, by = ELECTRODE_XY[b]
    return (ax + bx) / 2, (ay + by) / 2

def topography_map(ch_values, fig, ax, vmin=None, vmax=None):
    xx = np.array([channel_xy(c)[0] for c in ch_values])
    yy = np.array([channel_xy(c)[1] for c in ch_values])
    z = np.array(list(ch_values.values()))
    N, R = 200, 1.0
    xi = np.linspace(-R, R, N); yi = np.linspace(-R, R, N)
    zi = scipy.interpolate.griddata((xx, yy), z, (xi[None, :], yi[:, None]), method="cubic")
    for i in range(N):
        for j in range(N):
            if np.hypot(xi[i], yi[j]) > R:
                zi[j, i] = np.nan
    vmin = np.nanmin(zi) if vmin is None else vmin
    vmax = np.nanmax(zi) if vmax is None else vmax
    ax.contourf(xi, yi, zi, levels=np.linspace(vmin, vmax, 60), cmap="jet")
    ax.add_patch(mpatches.Circle((0, 0), R, fill=False, lw=2))
    ax.scatter(xx, yy, c="k", s=12)
    ax.set_xlim(-1.15, 1.15); ax.set_ylim(-1.15, 1.15); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])

def band_std_per_channel(spikegram_by_ch, band_idx):
    """std over time of one band's spikegram, per channel (normalized 0..1)."""
    vals = {ch: spec[band_idx].std() for ch, spec in spikegram_by_ch.items()}
    mx = max(vals.values()) or 1.0
    return {ch: v / mx for ch, v in vals.items()}

def band_std_fft(fft_by_ch, band_idx):
    vals = {ch: spec[band_idx].std() for ch, spec in fft_by_ch.items()}
    mx = max(vals.values()) or 1.0
    return {ch: v / mx for ch, v in vals.items()}

fig, axs = plt.subplots(2, len(bands), figsize=(20, 6))
for bi, band in enumerate(bands):
    topography_map(band_std_fft(fft_bins_by_ch, bi), fig, axs[0, bi])
    axs[0, bi].set_title(band, fontsize=14)
    topography_map(band_std_per_channel(spikegram_by_ch, bi), fig, axs[1, bi])
axs[0, 0].set_ylabel("FFT", fontsize=16); axs[1, 0].set_ylabel("SCTN", fontsize=16)
plt.suptitle("Topography map — FFT vs SCTN (seizure minute)", fontsize=16)
plt.tight_layout(); plt.show()

## 9. Topography — seizure vs non-seizure (paper Fig 7)

In [ ]:
seiz_sg = {ch: channel_spikegram("seizure", SEIZURE_MINUTE, ch) for ch in CHANNELS}
non_sg  = {ch: channel_spikegram("non_seizure", NON_SEIZURE_MINUTE, ch) for ch in CHANNELS}

fig, axs = plt.subplots(2, len(bands), figsize=(20, 6))
for bi, band in enumerate(bands):
    topography_map(band_std_per_channel(seiz_sg, bi), fig, axs[0, bi])
    axs[0, bi].set_title(band, fontsize=14)
    topography_map(band_std_per_channel(non_sg, bi), fig, axs[1, bi])
axs[0, 0].set_ylabel("Seizure", fontsize=16); axs[1, 0].set_ylabel("Non-seizure", fontsize=16)
plt.suptitle("Topography map — seizure vs non-seizure (SCTN)", fontsize=16)
plt.tight_layout(); plt.show()